# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR<sup>2</sup> dataset using the `mlcroissant` library. It follows a structured approach—loading metadata, exploring the schema, extracting data by `@id`, performing processing and analysis, and visualizing the results.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields and their `@id`s as described in the Croissant schema. This allows us to reference each entity unambiguously.

Let's explore the record sets, their fields, and, for each field, any column mappings. We'll reference everything by its `@id`.

In [ ]:
# Explore all record sets and their fields by `@id`
print('Available record sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  RecordSet name: {rs.metadata['name']} | @id: {rs.metadata['@id']}")
    if hasattr(rs, 'fields'):
        print('    Fields:')
        for field in rs.fields:
            print(f"      {field.metadata['name']} | field @id: {field.metadata['@id']}")
            if 'column' in field.metadata:
                columns = field.metadata['column']
                # Columns may be a list or single dict
                if isinstance(columns, list):
                    for col in columns:
                        print(f"         Column: {col['@id']}")
                elif isinstance(columns, dict):
                    print(f"         Column: {columns['@id']}")
    print()

## 3. Data Extraction

Now, we'll load data for each record set into a Pandas DataFrame, using only the record set `@id`s. Adjust the set of record sets and their IDs below if needed.

In [ ]:
# Get the `@id` of all record sets from previous cell
record_set_ids = [rs.metadata['@id'] for rs in dataset.record_sets]
print('RecordSet @ids:', record_set_ids)

# Load each record set into a DataFrame by @id
dataframes = dict()
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f'Loaded DataFrame for {rs_id}, shape={df.shape}')
    except Exception as e:
        print(f'Could not load records for {rs_id}: {e}')

# Display columns for the first record set (if available)
if len(dataframes) > 0:
    primary_rs_id = list(dataframes.keys())[0]
    print(f'Columns for primary record set ({primary_rs_id}):')
    print(dataframes[primary_rs_id].columns.tolist())
    display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate filtering, normalization, and grouping. We'll work with one record set and its numeric/categorical fields, all referenced by `@id`.

- **Note**: Change the values below if your DataFrame and fields differ. Use the output from Section 2 and 3.

In [ ]:
# Example: let's work with the first loaded record set
record_set_id = primary_rs_id
df = dataframes[record_set_id]

# Identify a numeric field (use field @id, as listed in previous step/output)
numeric_field_id = None
for col in df.columns:
    # Naive check: pick first float/int column
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print("No numeric field found in first record set DataFrame.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")
    # Example threshold (median for demonstration)
    threshold = df[numeric_field_id].median()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    # Normalize numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical column (by @id)
    group_field = None
    # Pick the first non-numeric, non-index column of object/string dtype
    for col in df.columns:
        if col == numeric_field_id: continue
        if df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field}:")
        display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and relationship with the chosen group field, all referenced by `@id`.

*Tip: The variables `numeric_field_id` and `group_field` are determined dynamically above. Modify as needed based on the actual field `@id` in your dataset.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is None:
    print('No numeric field available for visualization.')
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        order = df[group_field].value_counts().index
        sns.boxplot(x=df[group_field], y=df[numeric_field_id], order=order)
        plt.title(f'{numeric_field_id} grouped by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated loading and processing a Croissant-described dataset using only entity `@id` references with `mlcroissant`. This approach ensures robust, schema-compliant workflow for dataset exploration and reproducible analysis. You can adapt these steps for deeper modeling, additional visualizations, or richer data quality and bias audits as appropriate for your domain.

*Remember to always check the dataset documentation and schema JSON-LD for complete and up-to-date `@id` references when extending this workflow!*